# از هندسهٔ ساخته‌شده تا اندازه‌گیری کالیبره

ابتدا پیش‌بینی کنید: آیا کالیبراسیون کامل جابه‌جایی نامعلوم نمونه را حذف می‌کند؟ آیا هم‌بستگی تلرانس‌های ساخت می‌تواند اثر آن‌ها را خنثی کند؟ این دفترچه ابتدا حقیقت فرض‌شده را مشخص می‌کند و سپس مشاهدات مصنوعی می‌سازد.

**تمام اندازه‌گیری‌های اینجا مصنوعی‌اند.** مادهٔ لورنتس نمونه‌ای آموزشی است؛ تشدیدگر مدلی تک‌درگاهی با پارامترهای تعیین‌شده است، نه وصله‌ای ساخته‌شده و شبیه‌سازی‌شده. استانداردهای ایده‌آل اتصال‌کوتاه، بار و مدار باز، کالیبراسیون در صفحهٔ مشترک تک‌درگاهی را نشان می‌دهند؛ دستور کالیبراسیون شیپور در فضای آزاد نیستند.

از پایتون ۳٫۱۲ و requirements-labs.txt استفاده کنید و سلول‌ها را به‌ترتیب اجرا کنید. سلول آخر reference.json و synthetic-sweep.csv را در data/chapter-09 می‌نویسد؛ پیش از ویرایش نسخه‌ای از آن‌ها نگه دارید. برچسب نمودارها و خروجی‌های عددی در دو زبان مشترک‌اند.

In [ ]:
from pathlib import Path
from dataclasses import asdict, replace
import sys, json, hashlib, csv
import numpy as np
import matplotlib.pyplot as plt
ROOT=Path.cwd()
if not (ROOT/'simulations').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from simulations.materials_measurement import (Experiment, C0, lorentz, reflection,
    resonance, tolerance_samples, tolerance_linear_std, wilson, synthetic_sweep,
    uncertain_measurements, reference_factor, error_terms, measured, covariance_region)
p=Experiment()  # Edit physical and uncertainty assumptions here.
SCAN_OFFSETS_GHZ=(-1.5,1.5)  # Relative to the edited resonance; must bracket it.
SCAN_POINTS=241
plt.rcParams.update({'figure.dpi':120,'font.size':10,'axes.spines.top':False,
    'axes.spines.right':False,'axes.grid':True,'grid.alpha':.2})
colors=['#087f8c','#e18431','#7252a1']
evidence={'metadata':{'parameters':asdict(p),'model':'synthetic one-port calibration and assigned resonator',
    'source_sha256':hashlib.sha256((ROOT/'simulations/materials_measurement.py').read_text(encoding='utf-8').encode()).hexdigest()}}
pack=lambda a:np.column_stack([np.asarray(a).real,np.asarray(a).imag]).tolist()
print(asdict(p))
print('Resonant absorption:',1-abs(reflection(p.resonance_GHz,p=p))**2)

## ۱. ماده یک طیف است

قرارداد زمانی exp(−iωt) است. بخش موهومی مثبت گذردهی با محیط غیرفعال سازگار است. افزایش میرایی هر دو بخش حقیقی و موهومی را تغییر می‌دهد؛ در این مدل نوسانگر نمی‌توان اتلاف را مستقل از پاشندگی تغییر داد. ریشهٔ دوم اصلی، ضریب خاموشی مثبت دارد. این منحنی‌ها برازش ماده‌ای مشخص نیستند و به مدل جداگانهٔ تلرانس گذردهی مؤثر وارد نمی‌شوند.

In [ ]:
fm=np.linspace(6,20,281);material={}
fig,ax=plt.subplots(1,2,figsize=(11.5,4),layout='constrained')
for damping,color in zip((.6,1.2,2.4),colors):
    eps=lorentz(fm,damping_GHz=damping);index=np.sqrt(eps)
    material[str(damping)]={'epsilon':pack(eps),'index':pack(index)}
    ax[0].plot(fm,eps.real,color=color,label=f'Re, damping {damping} GHz')
    ax[0].plot(fm,eps.imag,'--',color=color)
    ax[1].plot(fm,index.real,color=color,label=f'n, damping {damping} GHz')
    ax[1].plot(fm,index.imag,'--',color=color)
ax[0].set(xlabel='Frequency (GHz)',ylabel='Relative permittivity',title='Illustrative Lorentz material: dashed = Im')
ax[1].set(xlabel='Frequency (GHz)',ylabel='Index / extinction coefficient',title='Principal sqrt(epsilon): dashed = extinction')
for a in ax:a.legend(fontsize=8)
plt.show()
evidence['materials']={'GHz':fm.tolist(),'rows':material,'epsilon_inf':2.,'strength':1.1,'pole_GHz':14.}

## ۲. تلرانس، هم‌بستگی و پذیرش

برای هر حالت ۲۰۰۰۰ نمونه با بذر ثابت تولید می‌شود. پهنای وصله و گذردهی مؤثر پراکندگی گاوسی دارند؛ تشدید از مقیاس‌گذاری محلی معکوس پهنا و معکوس ریشهٔ گذردهی در فصل پیروی می‌کند. پنل چپ سوگیری پهنا ندارد. در پنل راست سوگیری مشترک پهنا اضافه می‌شود و شرط |f₀−10 GHz| ≤ 0.12 GHz شمارش می‌شود. میله‌ها بازهٔ ویلسون ۹۵ درصد برای خطای نمونه‌گیری مونت‌کارلو در جمعیت فرضی‌اند، نه اطمینان به فرایندی واقعی. اعداد تصادفی مشترک مقایسه را کم‌نویزتر می‌کنند. هیچ نمونه‌ای بی‌صدا بریده نمی‌شود.

بازهٔ ۱۰ گیگاهرتز با تلرانس ۰٫۱۲ گیگاهرتز، حالت پیش‌فرض است. حدود هیستوگرام از نمونه‌های واقعی، ناحیهٔ رنگی از تشدید و تلرانس ویرایش‌شده و بازهٔ کالیبراسیون از جابه‌جایی‌های فرکانسی نسبت به تشدید ویرایش‌شده تعیین می‌شوند.


In [ ]:
biases=np.linspace(-.08,.08,9);tolerance={}
draw_sets={rho:tolerance_samples(rho,p=p) for rho in (-.6,0.,.6)}
# Include all sampled resonances and the full acceptance interval.
half_span=max(p.tolerance_GHz,*(float(np.max(abs(v-p.resonance_GHz))) for v in draw_sets.values()))*1.05
edges=np.linspace(p.resonance_GHz-half_span,p.resonance_GHz+half_span,75)
assert all(np.all((v>=edges[0])&(v<=edges[-1])) for v in draw_sets.values())
fig,ax=plt.subplots(1,2,figsize=(11.5,4),layout='constrained')
for rho,color in zip((-.6,0.,.6),colors):
    draws=draw_sets[rho];hist,_=np.histogram(draws,bins=edges,density=True)
    rows=[]
    for bias in biases:
        x=tolerance_samples(rho,float(bias),p=p);passed=int(np.count_nonzero(abs(x-p.resonance_GHz)<=p.tolerance_GHz))
        lo,hi=wilson(passed,len(x))
        rows.append({'bias_mm':float(bias),'acceptance':passed/len(x),'low':lo,'high':hi})
    tolerance[str(rho)]={'histogram':hist.tolist(),'std_GHz':float(draws.std(ddof=1)),
        'linear_std_GHz':float(tolerance_linear_std(rho,p)),'rows':rows}
    ax[0].stairs(hist,edges,color=color,label=f'rho = {rho:g}')
    y=np.array([v['acceptance'] for v in rows])
    ax[1].errorbar(biases*1000,y,yerr=[y-[v['low'] for v in rows],[v['high'] for v in rows]-y],
        fmt='o-',color=color,label=f'rho = {rho:g}')
ax[0].axvspan(p.resonance_GHz-p.tolerance_GHz,p.resonance_GHz+p.tolerance_GHz,color='#087f8c',alpha=.08)
ax[0].set(xlabel='Resonance (GHz)',ylabel='Probability density (1/GHz)',title='Zero bias; shaded = acceptance window',xlim=(edges[0],edges[-1]))
ax[1].set(xlabel='Common width bias (micrometres)',ylabel='Synthetic acceptance fraction',ylim=(0,1.03),
    title='20,000 samples; 95% Wilson intervals')
for a in ax:a.legend()
plt.show()
evidence['tolerance']={'edges_GHz':edges.tolist(),'rows':tolerance,'count':20000,'acceptance_center_GHz':p.resonance_GHz,'acceptance_tolerance_GHz':p.tolerance_GHz}
print({k:(v['std_GHz'],v['linear_std_GHz']) for k,v in tolerance.items()})

## ۳. کالیبراسیون و صفحهٔ مرجع

قرائت مختلط خام شامل خطای جهت‌مندی، ردیابی و تطبیق منبع است. مشاهدهٔ دقیق سه استاندارد ایده‌آل معلوم، جعبهٔ خطا را در هر فرکانس بازیابی می‌کند. پاسخ درگاه اصلاح می‌شود، اما مسیر هوایی ۵ میلی‌متری هنوز فاز را می‌چرخاند. با تقسیم بر عامل رفت‌وبرگشت معلوم به صفحهٔ سطح برسید. مشاهدهٔ صرف دامنه این تفاوت را پنهان می‌کند. بازیابی بدون نویز، جبر را نشان می‌دهد؛ در بخش بعد عدم‌قطعیت برمی‌گردد. فایل CSV قرائت خام، مشاهدهٔ سه استاندارد و حقیقت فرض‌شده را برای پردازش مستقل حفظ می‌کند.

In [ ]:
lo,hi=SCAN_OFFSETS_GHZ
if not np.isfinite([lo,hi]).all() or not lo<0<hi or p.resonance_GHz+lo<=0:
    raise ValueError('Use finite scan offsets bracketing the resonance, with positive frequencies.')
f=np.linspace(p.resonance_GHz+lo,p.resonance_GHz+hi,SCAN_POINTS)
assert f[0]<p.resonance_GHz<f[-1]
evidence['metadata']['scan_offsets_GHz']=list(SCAN_OFFSETS_GHZ)
sweep=synthetic_sweep(f,p)
fig,ax=plt.subplots(1,2,figsize=(11.5,4),layout='constrained')
for key,label,color,style in [('raw','raw reading',colors[1],'-'),('at_port','calibrated at port',colors[2],'-'),
                             ('recovered','corrected at surface',colors[0],'-')]:
    g=sweep[key];ax[0].plot(f,abs(g)**2,color=color,ls=style,label=label)
    ax[1].plot(f,np.rad2deg(np.unwrap(np.angle(g))),color=color,ls=style,label=label)
ax[0].plot(f,abs(sweep['truth'])**2,'k:',label='assigned truth')
ax[1].plot(f,np.rad2deg(np.unwrap(np.angle(sweep['truth']))),'k:',label='assigned truth')
ax[0].set(xlabel='Frequency (GHz)',ylabel='Magnitude squared (raw is not power)',title='Calibration changes the magnitude')
ax[1].set(xlabel='Frequency (GHz)',ylabel='Unwrapped phase (deg)',title='Plane correction changes phase')
for a in ax:a.legend(fontsize=8)
plt.show()
evidence['calibration']={'GHz':f.tolist(),**{k:pack(v) for k,v in sweep.items() if k!='condition'},
    'max_condition':float(max(sweep['condition']))}
print('Maximum complex error:',max(abs(sweep['recovered']-sweep['truth'])))

## ۴. عدم‌قطعیت یک ناحیه است

در هر یک از ۲۰۰۰ اجرای مونت‌کارلو، خطای شناخت استانداردها، نویز تازهٔ قرائت و مکان نمونه تولید می‌شود. بیضی از کوواریانس نمونه و آستانهٔ ۹۵ درصد توزیع نرمال دوبعدی یعنی ۵٫۹۹۱ استفاده می‌کند؛ پوشش در این مدل غیرخطی تقریبی است. علامت نارنجی حقیقت مصنوعی معلوم است؛ در آزمایش عادی چنین حقیقتی در دسترس نیست.

پنل راست **مدل تحلیلی جداگانه‌ای برای میانگین‌گیری** است: نویز دایروی در صفحهٔ سطح با RMS مختلط کل ۰٫۰۰۲ بین قرائت‌ها مستقل است، اما جابه‌جایی نمونه در هر دسته ثابت می‌ماند. بین دسته‌ها این جابه‌جایی گاوسی است. RMS مختلط نسبت به حقیقت، سوگیری غیرخطی کوچک را نیز دربر می‌گیرد. عدم‌قطعیت استاندارد کالیبراسیون در این منحنی ساده لحاظ نشده است. میانگین‌گیری یک اختلاف فاز مشترک نامعلوم را حذف نمی‌کند.

آستانهٔ ۵٫۹۹۱ برای ابر گاوسی دوبعدی با رتبهٔ کامل است. کوواریانس تباهیده به شکل نقطه یا خط نمایش داده می‌شود؛ برای خط، آستانهٔ گاوسی یک‌بعدی به کار می‌رود. پوشش در زیرفضای کوواریانس با مقدار ویژهٔ ناصفر و با کنترل باقیماندهٔ عمود محاسبه می‌شود.


In [ ]:
truth=complex(reflection(p.resonance_GHz,p=p));uncertainty={}
N=np.unique(np.geomspace(1,1000,80).astype(int))
for position in (0.,.05,.2):
    q=replace(p,position_std_mm=position);g,condition=uncertain_measurements(2000,q)
    xy=np.column_stack([g.real,g.imag]);region=covariance_region(xy)
    center,cov,ellipse=region['center'],region['covariance'],region['boundary']
    sigma_phase=4*np.pi*p.resonance_GHz*1e9*position*1e-3/C0
    floor2=2*abs(truth)**2*(-np.expm1(-sigma_phase**2/2))
    rms=np.sqrt(p.noise_rms**2/N+floor2)
    coverage=region['sample_fraction']
    uncertainty[str(position)]={'samples':xy[:400].tolist(),'ellipse':ellipse.tolist(),'region_rank':region['rank'],'region_kind':region['kind'],'region_threshold':region['threshold'],
        'center':center.tolist(),'covariance':cov.tolist(),'rms':float(np.sqrt(np.mean(abs(g-truth)**2))),
        'ellipse_sample_fraction':float(coverage),'averaging_rms':rms.tolist(),'floor':float(np.sqrt(floor2))}
fig,ax=plt.subplots(1,2,figsize=(11.5,4.2),layout='constrained')
baseline=uncertainty['0.05'];xy=np.array(baseline['samples']);ellipse=np.array(baseline['ellipse'])
ax[0].scatter(xy[:,0],xy[:,1],s=8,alpha=.25,color=colors[0],label='first 400 of 2,000 trials')
ax[0].plot(ellipse[:,0],ellipse[:,1],color=colors[2],label=f'approximate 95% {baseline["region_kind"]}')
ax[0].plot(truth.real,truth.imag,'x',ms=10,color=colors[1],label='synthetic truth')
ax[0].set(xlabel='Re(corrected reflection)',ylabel='Im(corrected reflection)',title='Placement standard deviation = 0.05 mm')
ax[0].axis('equal');ax[0].legend(fontsize=8)
for (key,row),color in zip(uncertainty.items(),colors):
    ax[1].loglog(N,row['averaging_rms'],color=color,label=f'placement std = {key} mm')
ax[1].set(xlabel='Independent readings per fixed-placement batch',ylabel='Complex RMS about truth',
    title='Separate analytic model: noise + fixed placement');ax[1].legend(fontsize=8)
plt.show()
evidence['uncertainty']={'rows':uncertainty,'readings':N.tolist(),'truth':[truth.real,truth.imag],'trials':2000}
print('Baseline complex RMS:',baseline['rms'],'Empirical ellipse fraction:',baseline['ellipse_sample_fraction'])

## ۵. مشاهدات را حفظ و مستقل بررسی کنید

فرمان python scripts/validate_chapter09.py را جدا اجرا کنید. توازن انرژی، رابطه‌های مستقل مد جفت‌شده و جعبهٔ خطا، استاندارد چهارم کنارگذاشته‌شده، انتقال فاز، همگرایی مشتق، مقایسهٔ مونت‌کارلو و کوواریانس خطی و ورودی‌های نامعتبر بررسی می‌شوند.

یکی از استانداردهای معلوم را با پاسخی تقریباً تکراری جایگزین کنید و شرطی‌بودن دستگاه را ببینید. خطای فاز مشترک استاندارد یا رانش هم‌بسته در فرکانس وارد و ناحیهٔ عدم‌قطعیت را مقایسه کنید. برای دادهٔ واقعی، پیش از برازش تشدید، تنظیمات ابزار، صفحه‌های مرجع، پاسخ مشخصه‌یابی‌شدهٔ استانداردها، واحدها و مشاهدات مختلط خام را نگه دارید. وقتی کانال‌های پراکندگی اندازه‌گیری‌نشده می‌توانند توان ببرند، عبارت 1−|Γspecular|² را جذب ننامید.

In [ ]:
out=ROOT/'data/chapter-09';out.mkdir(parents=True,exist_ok=True)
(out/'reference.json').write_text(json.dumps(evidence,separators=(',',':'),allow_nan=False)+'\n',encoding='utf-8')
d,t,s=error_terms(f)
standards=measured(np.array([-1,0,1]),d[:,None],t[:,None],s[:,None])
with (out/'synthetic-sweep.csv').open('w',newline='',encoding='utf-8') as stream:
    writer=csv.writer(stream)
    writer.writerow(['frequency_GHz','raw_re','raw_im','short_re','short_im','load_re','load_im',
                     'open_re','open_im','truth_re','truth_im','offset_mm'])
    for j,freq in enumerate(f):
        values=[freq,sweep['raw'][j].real,sweep['raw'][j].imag]
        for value in standards[j]:values.extend([value.real,value.imag])
        values.extend([sweep['truth'][j].real,sweep['truth'][j].imag,p.offset_mm])
        writer.writerow(values)
print('Saved synthetic observations and reference arrays; no experimental data is represented.')